In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

print("🚀 Executing Week 9 Conformance via Direct GUID ABFSS Storage Paths...")

try:
    # 1. READ DIRECTLY FROM BRONZE TABLES
    print("Reading raw Delta folders directly from Bronze storage...")
    
    INFO_BRONZE_PATH = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Bronze_Raw.Lakehouse/Tables/dbo/studentinfo_bronze"
    VLE_BRONZE_PATH  = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Bronze_Raw.Lakehouse/Tables/dbo/studentvle_bronze"
    
    df_info_raw = spark.read.format("delta").load(INFO_BRONZE_PATH)
    df_vle_raw  = spark.read.format("delta").load(VLE_BRONZE_PATH)
    
    # 2. CONFORM STUDENT REGISTRY PROFILES
    print("Conforming student profiles features...")
    df_student_silver = df_info_raw.select(
        F.col("id_student").cast(IntegerType()).alias("student_id"),
        F.trim(F.col("code_module")).alias("course_code"),
        F.trim(F.col("code_presentation")).alias("term_code"),
        F.trim(F.col("gender")).alias("biological_sex"),
        F.trim(F.col("region")).alias("geographic_region"),
        F.trim(F.col("highest_education")).alias("education_level"),
        F.when(F.col("imd_band").isNull() | (F.trim(F.col("imd_band")) == ""), "Unknown")
         .otherwise(F.trim(F.col("imd_band"))).alias("deprivation_band_pct"),
        F.trim(F.col("age_band")).alias("age_cohort"),
        F.col("num_of_prev_attempts").cast(IntegerType()).alias("previous_course_attempts"),
        F.col("studied_credits").cast(IntegerType()).alias("registered_credits"),
        F.trim(F.col("disability")).alias("disability_status"),
        F.trim(F.col("final_result")).alias("academic_outcome")
    ).withColumn("silver_processed_ts", F.current_timestamp())

    # WRITE DIRECTLY TO SILVER AUDITED STORAGE
    STUDENT_SILVER_WRITE_PATH = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Silver_Audited.Lakehouse/Tables/student_profiles_silver"
    
    print("Writing conformed profiles directly to Silver filesystem...")
    df_student_silver.write.mode("overwrite") \
                     .option("overwriteSchema", "true") \
                     .format("delta") \
                     .save(STUDENT_SILVER_WRITE_PATH)
    print(f"✔ student_profiles_silver written safely. Rows: {df_student_silver.count()}")

    # 3. CONFORM HIGH-VOLUME CLICKSTREAM LOGS
    print("Conforming fact clickstream logs...")
    df_vle_silver = df_vle_raw.select(
        F.col("id_student").cast(IntegerType()).alias("student_id"),
        F.trim(F.col("code_module")).alias("course_code"),
        F.trim(F.col("code_presentation")).alias("term_code"),
        F.col("id_site").cast(IntegerType()).alias("vle_material_id"),
        F.col("date").cast(IntegerType()).alias("relative_interaction_day"),
        F.col("sum_click").cast(IntegerType()).alias("daily_click_count")
    ).filter("student_id IS NOT NULL AND daily_click_count > 0") \
     .withColumn("silver_processed_ts", F.current_timestamp())

    ACTIVITY_SILVER_WRITE_PATH = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Silver_Audited.Lakehouse/Tables/student_activity_silver"

    print("Writing clickstream logs directly to Silver filesystem...")
    df_vle_silver.write.mode("overwrite") \
                 .option("overwriteSchema", "true") \
                 .format("delta") \
                 .save(ACTIVITY_SILVER_WRITE_PATH)
    print(f"✔ student_activity_silver written safely. Rows: {df_vle_silver.count()}")

    print("\n🚀 SUCCESS: Week 9 cross-lakehouse Silver conformance complete via direct filesystem injection!")

except Exception as e:
    print(f"❌ FILESYSTEM MOVEMENT FAILURE: {str(e)}")


StatementMeta(, 73ad5039-2554-488d-a332-08f25e3aaf89, 4, Finished, Available, Finished, False)

🚀 Executing Week 9 Conformance via Direct GUID ABFSS Storage Paths...
Reading raw Delta folders directly from Bronze storage...
Conforming student profiles features...
Writing conformed profiles directly to Silver filesystem...
✔ student_profiles_silver written safely. Rows: 32593
Conforming fact clickstream logs...
Writing clickstream logs directly to Silver filesystem...
✔ student_activity_silver written safely. Rows: 10655280

🚀 SUCCESS: Week 9 cross-lakehouse Silver conformance complete via direct filesystem injection!


In [6]:
INFO_SILVER_PATH = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Silver_Audited.Lakehouse/Tables/student_profiles_silver"
ACTIVITY_SILVER_PATH = "abfss://OULAD_DENG_HCAI_RESEARCH@onelake.dfs.fabric.microsoft.com/OULAD_DENG_Silver_Audited.Lakehouse/Tables/student_activity_silver"

StatementMeta(, 73ad5039-2554-488d-a332-08f25e3aaf89, 8, Finished, Available, Finished, False)

In [7]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS student_profiles_silver
    USING DELTA
    LOCATION '{INFO_SILVER_PATH}'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS student_activity_silver
    USING DELTA
    LOCATION '{ACTIVITY_SILVER_PATH}'
""")


StatementMeta(, 73ad5039-2554-488d-a332-08f25e3aaf89, 9, Finished, Available, Finished, False)

DataFrame[]

In [8]:
spark.sql("SHOW TABLES").show()

StatementMeta(, 73ad5039-2554-488d-a332-08f25e3aaf89, 10, Finished, Available, Finished, False)

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|OULAD_DENG_HCAI_R...|student_activity_...|      false|
|OULAD_DENG_HCAI_R...|student_profiles_...|      false|
+--------------------+--------------------+-----------+



In [10]:
%%sql
SELECT * FROM student_profiles_silver LIMIT 10

StatementMeta(, 73ad5039-2554-488d-a332-08f25e3aaf89, 12, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 13 fields>